In [1]:
# ============================================================
# PREPARACIÓN DE POBLACIÓN INEI → CSV TRIMESTRALES
# Abre el Excel del INEI (subido a Colab) y genera dos CSV
# de población trimestral por departamento y provincia.
# ============================================================

import pandas as pd
import numpy as np

In [2]:
# 1. Carga de datos
# ⬇Nombre del Excel
ARCHIVO_INEI = "poblacion_inei.xlsx"
# Carga cruda
# El Excel del INEI trae 6 filas de títulos antes de los datos.
años = [str(a) for a in range(2018, 2027)]  # 2018..2026
df = pd.read_excel(
    ARCHIVO_INEI, sheet_name=0, header=None, skiprows=6,
    names=['UBIGEO', 'Nombre'] + años
)
print(f"Filas leídas: {df.shape[0]:,}")

Filas leídas: 2,139


In [3]:
# 2. Limpieza
df = df.dropna(subset=['UBIGEO'])
df['UBIGEO'] = (df['UBIGEO'].astype(str).str.strip()
                .str.replace('.0', '', regex=False).str.zfill(6))
for a in años:
    df[a] = pd.to_numeric(df[a], errors='coerce')
df = df[df['UBIGEO'] != '000000']   # quitar total PERÚ

In [4]:
# 3. Clasificar nivel geográfico
df['cod_dep']  = df['UBIGEO'].str[:2]
df['cod_prov'] = df['UBIGEO'].str[:4]
df['nivel'] = np.where(df['UBIGEO'].str[2:] == '0000', 'DEP',
              np.where(df['UBIGEO'].str[4:] == '00', 'PROV', 'DIST'))
print("\nConteo por nivel:")
print(df['nivel'].value_counts())


Conteo por nivel:
nivel
DIST    1916
PROV     196
DEP       25
Name: count, dtype: int64


In [5]:
# 4. Verificación anti doble-conteo
dist = df[df['nivel'] == 'DIST'].copy()
print(f"\nDistritos: {len(dist):,}")
print(f"Suma distritos 2019: {dist['2019'].sum():,.0f}  (esperado 32,131,400)")
print(f"Departamentos: {dist['cod_dep'].nunique()} | Provincias: {dist['cod_prov'].nunique()}")


Distritos: 1,916
Suma distritos 2019: 32,131,400  (esperado 32,131,400)
Departamentos: 36 | Provincias: 220


In [6]:
# 5. Anual a trimestral por interpolación lineal
# P(t) = P(año) + (t-1)/4 * ( P(año+1) - P(año) ) — sin tasa asumida
def construir_trimestral(dist_df, col_geo, nombre_col):
    pop_anual = dist_df.groupby(col_geo)[años].sum().reset_index()
    pop_long = pop_anual.melt(id_vars=col_geo, var_name='Anho', value_name='Pob_anual')
    pop_long['Anho'] = pop_long['Anho'].astype(int)
    pop_long = pop_long.sort_values([col_geo, 'Anho'])
    pop_long['Pob_sig'] = pop_long.groupby(col_geo)['Pob_anual'].shift(-1)
    pop_long['Pob_sig'] = pop_long['Pob_sig'].fillna(pop_long['Pob_anual'])
    filas = []
    for _, r in pop_long.iterrows():
        for t in [1, 2, 3, 4]:
            pob = r['Pob_anual'] + (t - 1)/4 * (r['Pob_sig'] - r['Pob_anual'])
            filas.append({col_geo: r[col_geo], 'Anho': int(r['Anho']),
                          'Trimestre': t, nombre_col: round(pob, 1)})
    return pd.DataFrame(filas)

pop_dep_trim  = construir_trimestral(dist, 'cod_dep',  'Poblacion_dep_trim')
pop_prov_trim = construir_trimestral(dist, 'cod_prov', 'Poblacion_prov_trim')

In [7]:
# 6. Guardar los dos CSV
pop_dep_trim.to_csv('poblacion_dep_trimestral_INEI.csv', index=False)
pop_prov_trim.to_csv('poblacion_prov_trimestral_INEI.csv', index=False)
print("\n✓ Generados: poblacion_dep_trimestral_INEI.csv y poblacion_prov_trimestral_INEI.csv")
print("\nEjemplo Amazonas 2020:")
print(pop_dep_trim[(pop_dep_trim['cod_dep']=='01') & (pop_dep_trim['Anho']==2020)].to_string(index=False))


✓ Generados: poblacion_dep_trimestral_INEI.csv y poblacion_prov_trimestral_INEI.csv

Ejemplo Amazonas 2020:
cod_dep  Anho  Trimestre  Poblacion_dep_trim
     01  2020          1            426806.0
     01  2020          2            427232.5
     01  2020          3            427659.0
     01  2020          4            428085.5
